In [25]:
import torch
words = open('names.txt', 'r').read().splitlines()
print(torch.__version__, len(words), words[:5])

2.15.0 32033 ['emma', 'olivia', 'ava', 'isabella', 'sophia']


In [26]:
text = ''.join(words)
print(text[:50])
print(len(text))
chars = sorted(set(text))
print(chars)
print(len(chars))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
print(stoi)
print(stoi['a'], stoi['z'], stoi['.'])   # should print: 1 26 0
itos = {i+1: s for i, s in enumerate(chars)}
itos[0] = '.'
print(itos[1], itos[26], itos[0])   # should print: a z .

emmaoliviaavaisabellasophiacharlottemiaameliaharpe
196113
['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
26
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
1 26 0
a z .


In [27]:
import torch
N = torch.zeros((3, 3), dtype=torch.int32)   # a 3x3 table of zeros (whole numbers)
N[1, 2] += 1                                  # add 1 to row 1, column 2
print(N)

chs = ['.'] + list('emma') + ['.']            # ['.', 'e', 'm', 'm', 'a', '.']
for ch1, ch2 in zip(chs, chs[1:]):            # pairs each item with the next one
    print(ch1, ch2)
    

tensor([[0, 0, 0],
        [0, 0, 1],
        [0, 0, 0]], dtype=torch.int32)
. e
e m
m m
m a
a .


In [28]:
# 1. Create a 27x27 table of zeros called N
import torch
N = torch.zeros((27,27), dtype=torch.int32)

# 2. Loop over every word in words
for word in words:
    chs = ['.'] + list(word) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1,ix2] += 1
        
    
# 3.   Make chs: '.' + the letters of the word + '.'
# 4.   Loop over every pair (ch1, ch2) of neighbors
# 5.     Turn ch1 and ch2 into numbers using stoi
# 6.     Add 1 to N at that row and column

In [29]:
P = (N+1).float()
P = P / P.sum(1, keepdim=True)
print(P[0].sum())    # should be 1.0
print(P.sum(1))      # 27 values, all ~1.0

tensor(1.)
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [30]:
# picks index 0 about 60% of the time, 1 about 30%, 2 about 10%
# .item() turns a 1-element tensor into a plain Python number

# 1. Start at the '.' row (ix = 0) and make an empty list for the letters
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    ix = 0
    out = []
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))
# 2. Loop forever:
# 3.   Take the row of probabilities for the current letter: P[ix]
# 4.   Sample the next index from that row
# 5.   If it's 0 (the '.'), the name is over: break
# 6.   Otherwise, turn ix into a letter with itos and add it to the list
# 7. Join the letters into a string and print it

cexze
momasurailezitynn
konimittain
llayn
ka
da
staiyaubrtthrigotai
moliellavo
ke
teda


In [31]:
# 1. Start two counters: log_likelihood = 0.0 and n = 0
log_likelihood = 0.0
n = 0
for word in words:
    chs = ['.'] + list(word) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1,ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1

nll = -log_likelihood
print(nll / n)




tensor(2.4544)


Neural Network Version

In [ ]:
xs = []
ys = []
for word in words[:1]:
    chs = ['.'] + list(word) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
W = torch.randn((27, 27), generator=g, requires_grad=True)

import torch.nn.functional as F


xenc = F.one_hot(xs, num_classes=27).float()
(W**2).mean()
logits = xenc @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)


loss = -probs[torch.arange(5), ys].log().mean() + 0.01*(W**2).mean()


# backward pass
W.grad = None
loss.backward()

W.data += -0.1 * W.grad



